# День 4 — Baseline без обучения трансформера

Цель: получить CLS-эмбеддинги готовой модели и обучить на них
логистическую регрессию. Трансформер не дообучается.

## Задача 1: токенизация текстов

`tokenize_texts` уже написана в Дне 1 (`sentiment_lab.tokenization`).

In [ ]:
from transformers import AutoTokenizer

from sentiment_lab.tokenization import tokenize_texts

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

texts = [
    "This movie was great!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it."
]
tokens = tokenize_texts(tokenizer, texts)
print(f'Shape: {tokens["input_ids"].shape}')
print(f'Attention mask:\n{tokens["attention_mask"]}')

## Задача 2: CLS-эмбеддинги

Функция из Дня 2 (`get_embeddings`) делает то же, что в задаче названо
`get_cls_embeddings`: батчи по `batch_size`, `no_grad`, `last_hidden_state[:, 0, :]`, 
`cpu().numpy()` и `np.vstack`.

In [ ]:
from sentiment_lab.embeddings import get_embeddings, load_model

model = load_model(model_name)
embeddings = get_embeddings(texts, tokenizer, model)
print(f'Embeddings shape: {embeddings.shape}')

## Задача 3: Logistic Regression на эмбеддингах

Датасет: `data/imdb_train.csv` (колонки `text`, `label`). Если файла нет:
`uv run python scripts/download_data.py --max-rows 5000`.

Для скорости на CPU берём 500 отзывов; увеличение `N_SAMPLES` до 5000 улучшит качество.

In [ ]:
import pandas as pd

df = pd.read_csv("data/imdb_train.csv")
N_SAMPLES = 500
df = df.sample(n=N_SAMPLES, random_state=42)

texts_all = df["text"].tolist()
y = df["label"].to_numpy()
print(f'Текстов: {len(texts_all)}, классов: {sorted(set(y.tolist()))}')

In [ ]:
X = get_embeddings(texts_all, tokenizer, model)
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
from sentiment_lab.baseline import save_results, train_baseline

result = train_baseline(X, y)
print(result.report)
print(f'Macro F1: {result.macro_f1:.3f}')

path = save_results(result)
print(f'Результат сохранён: {path}')